# Kuhn Poker #

We will solve Kuhn Poker using LPs! Run the cell below first in order to import the necessary libraries.

In [1]:
import numpy as np
import pulp

We need to build the payoff matrix in order to solve the game. Each player has a different number of strategies. Each strategy will be represented by a $6$ character string. The first $2$ characters represent the two actions (in order) taken for the Jack. The next two charcters are for the Queen. The last two characters are for the King.

C = Check/Call

F = Fold

R = Raise

X = No Remaining Action Possible

In [2]:
#Evaluation Function. This determines which player wins a hand
def evaluate(s1, s2, cards):

  '''
  Given player 1 using strategy 1 and player 2 using strategy 2, this function will return the payoff to player 1.

  Inputs:
    s1: string representing player 1's strategy
    s2: string representing player 2's strategy
    cards: A string with the cards for each player (p1Card,p2Card)

  Outputs:
    payoff: The payoff for player 1
  '''

  def showdown(cards):

    '''

    Determines the winner of a showdown.

    Inputs:
      cards: A string with the cards for each player (p1Card,p2Card)
    Outputs:
      winner: The winner of the showdown: 1 for player 1, -1 for player 2

    '''

    p1Card = cards[0]
    p2Card = cards[1]

    if p1Card == 'J':
      return -1
    elif p1Card == 'K':
      return 1
    elif p1Card == 'Q':
      if p2Card == 'J':
        return 1
      else:
        return -1

    #Get the two characters of the string corresponding to each player's cards
  p1Card = cards[0]
  p2Card = cards[1]

  #For player 1
  if p1Card == 'J':
    substrat1 = s1[0:2]
  elif p1Card == 'Q':
    substrat1 = s1[2:4]
  else:
    substrat1 = s1[4:6]

  #For player 2
  if p2Card == 'J':
    substrat2 = s2[0:2]
  elif p2Card == 'Q':
    substrat2 = s2[2:4]
  else:
    substrat2 = s2[4:6]

  #Go through the action and determine the winner

  #Player 1's Action First
  if substrat1[0] == 'C':
    if substrat2[0] == 'C':
      payoff = showdown(cards)

    elif substrat2[0] == 'R':
      if substrat1[1] == 'C':
        payoff = 2*showdown(cards)
      else:
        payoff = -1

  else:
    if substrat2[1] == 'C':
      payoff = 2*showdown(cards)
    else:
      payoff = 1

  return payoff

#Get the card list for future reference
cardlist = ['JQ','JK','QJ', 'QK', 'KJ', 'KQ']

#Get the possible strategies for each player
substrat1 = ['CC','CF','RX']
substrat2 = ['CC','CF','RC','RF']

strats1 = [a + b + c for a in substrat1 for b in substrat1 for c in substrat1]
strats2 = [a + b + c for a in substrat2 for b in substrat2 for c in substrat2]

#Payoff Matrix
A = np.zeros((len(strats1),len(strats2)))

#Find the expectations for the matrix A
for i in range(len(strats1)):
  for j in range(len(strats2)):
    for cards in cardlist:
      A[i,j] += evaluate(strats1[i],strats2[j],cards)

A/=6

n,m = A.shape

Now, we can formulate the LP. Recall how we formulate an LP to solve for an optimal mixed strategy (Lecture 27).

In [3]:
#Create the LP

kuhn1 = pulp.LpProblem('kuhn1',pulp.LpMaximize)

#Create the variables
strat1vars = pulp.LpVariable.dicts('Strategy1: ', strats1, lowBound=0, upBound=1, cat='Continuous')
w = pulp.LpVariable('w', cat='Continuous')

#Get the objective
kuhn1 += w

#Get the constraints
#For each column, we get the corresponding constraint
for j in range(m):
  kuhn1 += pulp.lpSum([A[i,j]*strat1vars[strats1[i]] for i in range(n)]) >= w

#Get the sum of probabilities constraint
kuhn1 += pulp.lpSum([strat1vars[strats1[i]] for i in range(n)]) == 1

kuhn1.writeLP("kuhn1.lp")
kuhn1.solve()

#Check if the solution is optimal/infeasible
print("PLayer 1:")
print(pulp.LpStatus[kuhn1.status])

#Get the variable values
varvalues = {}
for x in kuhn1.variables():
    varvalues[x.name] = x.varValue
    if x.varValue >0:
      print(x.name, "=", x.varValue)

print(f"Optimal Expected Payout: {w.varValue}")

print('\n\n\n')

kuhn2 = pulp.LpProblem('kuhn2',pulp.LpMinimize)

#Create the variables
strat2vars = pulp.LpVariable.dicts('Strategy2: ', strats2, lowBound=0, upBound=1, cat='Continuous')
v = pulp.LpVariable('v', cat='Continuous')

#Get the objective
kuhn2 += v

#Get the constraints
#For each row, we get the corresponding constraint
for i in range(n):
  kuhn2 += pulp.lpSum([A[i,j]*strat2vars[strats2[j]] for j in range(m)]) <= v

#Get the sum of probabilities constraint
kuhn2 += pulp.lpSum([strat2vars[strats2[i]] for i in range(m)]) == 1

kuhn2.writeLP("kuhn2.lp")
kuhn2.solve()

#Check if the solution is optimal/infeasible
print("PLayer 2:")
print(pulp.LpStatus[kuhn2.status])

#Get the variable values
varvalues = {}
for x in kuhn2.variables():
    varvalues[x.name] = x.varValue
    if x.varValue >0:
      print(x.name, "=", x.varValue)
print(f"Optimal Expected Payout: {v.varValue}")


print('\n\n\n')

print(f"The theoretical expected payout under optimal play should be -1/18 = {-1/18}")

PLayer 1:
Optimal
Strategy1:__CFCCCC = 0.33333333
Strategy1:__CFCFCC = 0.66666667
Optimal Expected Payout: -0.055555556




PLayer 2:
Optimal
Strategy2:__CFCFRC = 0.66666667
Strategy2:__RFCCRC = 0.33333333
Optimal Expected Payout: -0.055555556




The theoretical expected payout under optimal play should be -1/18 = -0.05555555555555555
